In [ ]:
# ============================================================
# CELL 1: Install Dependencies (Colab-friendly)
# ============================================================

!pip install transformers==4.41.0
!pip install bitsandbytes==0.41.3  # For 4-bit quantization
!pip install sentence-transformers  # For embeddings

# ============================================================
# CELL 2: Imports
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict
from statistics import mean, stdev

import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

import json
import re
import hashlib
import random

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    AutoModel
)

import pandas as pd

print("✅ All imports completed")


# ============================================================
# CELL 3: Class 1 - NeuralActivityMonitor (Hooks & Measurements)
# ============================================================

class NeuralActivityMonitor:
    """
    Simulates neuroimaging and electrophysiological recordings.
    Uses hook-based instrumentation similar to fMRI and multi-electrode arrays.
    """

    def __init__(self, model: nn.Module):
        self.activations = {}
        self.gradients = {}
        self.hooks = []

        # Register hooks on all Linear layers
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                self.hooks.append(
                    module.register_forward_hook(self._create_activation_hook(name))
                )
                self.hooks.append(
                    module.register_full_backward_hook(self._create_gradient_hook(name))
                )

    def _create_activation_hook(self, name: str):
        """Creates forward hook to record activation statistics"""
        def hook(module, inp, out):
            self.activations[name] = {
                "mean": out.mean().item(),
                "std": out.std().item(),
                "dead": float((out.abs() < 1e-3).float().mean()),
                "max": out.max().item(),
                "min": out.min().item()
            }
        return hook

    def _create_gradient_hook(self, name: str):
        """Creates backward hook to record gradient statistics"""
        def hook(module, grad_in, grad_out):
            if grad_out[0] is not None:
                g = grad_out[0]
                self.gradients[name] = {
                    "norm": g.norm().item(),
                    "vanishing": float((g.abs() < 1e-7).float().mean()),
                    "max": g.max().item(),
                    "min": g.min().item()
                }
        return hook

    def clear(self):
        """Clear stored activations and gradients"""
        self.activations.clear()
        self.gradients.clear()

    def remove(self):
        """Remove all hooks"""
        for hook in self.hooks:
            hook.remove()

    def get_activation_summary(self) -> str:
        """Get formatted activation summary for LLM prompts"""
        lines = []
        for name, stats in self.activations.items():
            lines.append(f"  Layer {name}: mean={stats['mean']:.3f}, dead={stats['dead']:.2%}")
        return "\n".join(lines) if lines else "  No activation data"

    def get_gradient_summary(self) -> str:
        """Get formatted gradient summary for LLM prompts"""
        lines = []
        for name, stats in self.gradients.items():
            lines.append(f"  Layer {name}: norm={stats['norm']:.3e}, vanishing={stats['vanishing']:.2%}")
        return "\n".join(lines) if lines else "  No gradient data"


# ============================================================
# CELL 4: Class 2 - DiagnosticEngine (Physics-Inspired Metrics)
# ============================================================

class DiagnosticEngine:
    """
    Physics-inspired diagnostic framework for neural network training dynamics.
    Implements Equations 9-11, 13, 16, 21-22 from the paper.
    """

    def __init__(self, history_window: int = 10):
        self.training_history = []
        self.history_window = history_window

        # State tracking for diagnostics
        self.prev_energy = None
        self.prev_grad_norms = []
        self.gradient_history = []

    def record_epoch(self, epoch: int, loss: float, activity: Dict):
        """Record epoch data for trend analysis"""
        self.training_history.append({
            "epoch": epoch,
            "loss": loss,
            "activity": activity
        })

    def compute_stiffness(self, gradients: Dict) -> float:
        """
        Equation 13: Stiffness Ratio
        S = max(||∇L||) / min(||∇L||)
        """
        grad_norms = []
        for v in gradients.values():
            if isinstance(v, dict) and 'norm' in v:
                grad_norms.append(v['norm'])
            elif isinstance(v, (int, float)):
                grad_norms.append(v)

        if not grad_norms:
            return 1.0

        return np.max(grad_norms) / (np.min(grad_norms) + 1e-8)

    def compute_hamiltonian_energy(self, loss: float, gradients: Dict) -> Dict:
        """
        Equations 9-11: Hamiltonian Energy Evolution
        H = U + T where U = loss, T = ½||∇L||²
        """
        grad_norms = []
        for v in gradients.values():
            if isinstance(v, dict) and 'norm' in v:
                grad_norms.append(v['norm'])
            elif isinstance(v, (int, float)):
                grad_norms.append(v)

        potential = loss
        kinetic = 0.5 * np.mean(np.array(grad_norms)**2) if grad_norms else 0.0
        total_energy = potential + kinetic

        if self.prev_energy is not None:
            energy_change = total_energy - self.prev_energy
            if energy_change > 0.05 * total_energy:
                regime = "explosive"
            elif energy_change < -0.05 * total_energy:
                regime = "overdamped"
            else:
                regime = "conservative"
        else:
            energy_change = 0.0
            regime = "initializing"

        self.prev_energy = total_energy

        return {
            'total_energy': total_energy,
            'potential': potential,
            'kinetic': kinetic,
            'energy_change': energy_change,
            'regime': regime,
            'kinetic_potential_ratio': kinetic / (potential + 1e-8)
        }

    def compute_critical_slowing(self, gradients: Dict) -> bool:
        """
        Equation 16: Critical Slowing Detection
        Returns True if gradients change < 1% between epochs
        """
        grad_norms = []
        for v in gradients.values():
            if isinstance(v, dict) and 'norm' in v:
                grad_norms.append(v['norm'])
            elif isinstance(v, (int, float)):
                grad_norms.append(v)

        if not grad_norms:
            return False

        if len(self.prev_grad_norms) > 0:
            delta = np.abs(np.array(grad_norms) - np.array(self.prev_grad_norms))
            relative_change = np.mean(delta) / (np.mean(grad_norms) + 1e-8)
            critical = relative_change < 0.01
        else:
            critical = False

        self.prev_grad_norms = grad_norms
        return critical

    def predict_convergence(self, gradients: Dict) -> float:
        """
        Equations 21-22: Convergence Prediction
        Estimates remaining epochs until convergence
        """
        grad_norms = []
        for v in gradients.values():
            if isinstance(v, dict) and 'norm' in v:
                grad_norms.append(v['norm'])
            elif isinstance(v, (int, float)):
                grad_norms.append(v)

        if not grad_norms:
            return np.inf

        mean_grad_norm = np.mean(grad_norms)
        self.gradient_history.append(mean_grad_norm)

        if len(self.gradient_history) > 10:
            self.gradient_history.pop(0)

        if len(self.gradient_history) < 2:
            return np.inf

        decay_rates = []
        for i in range(1, len(self.gradient_history)):
            ratio = self.gradient_history[i] / (self.gradient_history[i-1] + 1e-8)
            if ratio > 0:
                decay_rates.append(-np.log(ratio))

        if not decay_rates:
            return np.inf

        beta = np.mean(decay_rates)
        if beta <= 0:
            return np.inf

        target_gradient = 1e-6
        initial_gradient = self.gradient_history[0]

        if initial_gradient <= target_gradient:
            return 0

        return (1.0 / beta) * np.log(initial_gradient / target_gradient)

    def get_diagnostics(self, loss: float, gradients: Dict) -> Dict:
        """Collect all diagnostic metrics"""
        stiffness = self.compute_stiffness(gradients)
        hamiltonian = self.compute_hamiltonian_energy(loss, gradients)
        critical_slowing = self.compute_critical_slowing(gradients)
        t_converge = self.predict_convergence(gradients)

        # Phase classification
        if hamiltonian['regime'] == "explosive":
            phase = "explosive"
        elif stiffness > 100:
            phase = "stiff"
        elif critical_slowing:
            phase = "critical_slowing"
        elif hamiltonian['regime'] == "overdamped":
            phase = "overdamped"
        elif t_converge < 50:
            phase = "fast_convergence"
        elif t_converge < 200:
            phase = "stable_convergence"
        else:
            phase = "slow_convergence"

        return {
            'stiffness': stiffness,
            'critical_slowing': critical_slowing,
            't_converge': t_converge,
            'phase': phase,
            'hamiltonian': hamiltonian
        }

    def summarize_trends(self) -> str:
        """Summarize recent training trends"""
        hist = self.training_history[-self.history_window:]
        if len(hist) < 2:
            return "Insufficient history."

        start, end = hist[0]["loss"], hist[-1]["loss"]
        delta = end - start

        return f"Loss start: {start:.4f}\nLoss end: {end:.4f}\nΔLoss: {delta:.4f}"


# ============================================================
# CELL 5: Class 3 - LLMNeuroscientist (Analysis & Interpretation)
# ============================================================

class LLMNeuroscientist:
    """
    Synthetic neuroscientist using LLMs to interpret training dynamics.
    Combines LLM generation with rule-based fallback for robustness.
    """

    def __init__(self, model_name: str = "google/flan-t5-large", use_4bit: bool = True):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = model_name

        # Set deterministic seeds
        self._set_deterministic(42)

        # Load LLM with appropriate configuration
        self.tokenizer = None
        self.model = None
        self._load_model(use_4bit)

        # Initialize diagnostic engine
        self.diagnostics = DiagnosticEngine()

    def _set_deterministic(self, seed: int):
        """Set all random seeds for reproducibility"""
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)
        random.seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def _load_model(self, use_4bit: bool):
        """Load LLM with appropriate configuration"""
        try:
            # For Flan-T5 (Seq2Seq) - faster for Colab
            if "flan" in self.model_name.lower():
                self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
                self.model = AutoModelForSeq2SeqLM.from_pretrained(
                    self.model_name,
                    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
                ).to(self.device)
                self.is_seq2seq = True
                print(f"✅ Loaded Seq2Seq model: {self.model_name}")

            # For causal LMs (Zephyr, TinyLlama, GPT-2)
            else:
                if use_4bit and torch.cuda.is_available():
                    bnb_config = BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.float16
                    )
                else:
                    bnb_config = None

                self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
                if self.tokenizer.pad_token is None:
                    self.tokenizer.pad_token = self.tokenizer.eos_token

                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_name,
                    quantization_config=bnb_config,
                    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                    device_map="auto" if torch.cuda.is_available() else None
                )
                self.is_seq2seq = False
                print(f"✅ Loaded Causal LM: {self.model_name}")

            self.model.eval()

        except Exception as e:
            print(f"❌ LLM load failed: {e}")
            self.model = None

    def record_epoch(self, epoch: int, loss: float, activations: Dict):
        """Record epoch for diagnostic tracking"""
        self.diagnostics.record_epoch(epoch, loss, activations)

    def analyze(self, epoch: int, loss: float, activations: Dict, gradients: Dict) -> str:
        """Generate analysis for current training state"""
        self.record_epoch(epoch, loss, activations)

        # Try LLM first
        if self.model is not None:
            try:
                prompt = self._build_prompt(epoch, loss, activations, gradients)
                response = self._generate(prompt)
                # LOWER THRESHOLD - accept shorter responses
                if len(response.split()) >= 10:  # Was 15, now 10
                    return response
                else:
                    print(f"Response too short ({len(response.split())} words), using rule-based")
            except Exception as e:
                print(f"LLM analysis failed: {e}")

        # Fallback to rule-based analysis
        return self._rule_based_analysis(epoch, loss, activations, gradients)

    def _build_prompt(self, epoch: int, loss: float, activations: Dict, gradients: Dict) -> str:
          """
          Professional scientific prompt for neural network training analysis.
          Designed to produce publication-quality interpretations.
          """

          diag = self.diagnostics.get_diagnostics(loss, gradients)

          # Format activation statistics with more detail
          act_details = []
          for name, stats in activations.items():
              if isinstance(stats, dict):
                  mean_val = stats.get('mean', 0)
                  dead_val = stats.get('dead', 0)
                  # Interpret activation health
                  if mean_val > 0.3:
                      act_health = "healthy"
                  elif mean_val > 0.1:
                      act_health = "moderate"
                  else:
                      act_health = "suppressed"
                  act_details.append(f"Layer {name}: mean={mean_val:.3f} ({act_health}), dead={dead_val:.1%}")

          act_str = "\n  ".join(act_details) if act_details else "No activation data"

          # Format gradient statistics with interpretation
          grad_details = []
          for name, stats in gradients.items():
              if isinstance(stats, dict):
                  norm_val = stats.get('norm', 0)
                  vanish_val = stats.get('vanishing', 0)
                  # Interpret gradient health
                  if norm_val > 0.01:
                      grad_health = "strong flow"
                  elif norm_val > 0.001:
                      grad_health = "moderate flow"
                  elif norm_val > 1e-5:
                      grad_health = "weak flow"
                  else:
                      grad_health = "vanishing"
                  grad_details.append(f"Layer {name}: norm={norm_val:.3e} ({grad_health}), vanishing={vanish_val:.1%}")

          grad_str = "\n  ".join(grad_details) if grad_details else "No gradient data"

          # Interpret stiffness
          stiffness = diag['stiffness']
          if stiffness > 100:
              stiffness_interpret = "severe imbalance - layers learning at very different rates"
          elif stiffness > 20:
              stiffness_interpret = "moderate imbalance - consider adjusting layer-wise learning rates"
          elif stiffness > 5:
              stiffness_interpret = "mild imbalance - acceptable for this architecture"
          else:
              stiffness_interpret = "balanced - healthy gradient flow across layers"

          # Interpret energy regime
          energy_regime = diag['hamiltonian']['regime']
          if energy_regime == "explosive":
              regime_interpret = "UNSTABLE - energy increasing, learning rate likely too high"
          elif energy_regime == "overdamped":
              regime_interpret = "SLUGGISH - energy dissipating too quickly, learning rate may be too low"
          elif energy_regime == "conservative":
              regime_interpret = "STABLE - healthy energy evolution"
          else:
              regime_interpret = "initializing"

          # Interpret critical slowing
          critical = diag['critical_slowing']
          if critical:
              critical_interpret = "DETECTED - gradients changing slowly, possible stagnation near flat region"
          else:
              critical_interpret = "not detected - optimization actively exploring landscape"

          # Predict convergence if loss is improving
          t_converge = diag['t_converge']
          if t_converge < float('inf') and t_converge > 0:
              converge_note = f"Predicted convergence in approximately {t_converge:.0f} additional epochs"
          elif loss < 0.05:
              converge_note = "Network appears to have converged"
          else:
              converge_note = "Convergence prediction unavailable - insufficient gradient history"

          # Build the professional prompt
          prompt = f"""You are a computational neuroscientist analyzing neural network training dynamics. Provide a concise, technical analysis based ONLY on the data below.

      ================================================================================
      TRAINING SNAPSHOT AT EPOCH {epoch}
      ================================================================================

      LOSS METRICS:
        Current loss: {loss:.6f}
        {converge_note}

      DIAGNOSTIC METRICS:
        Stiffness ratio: {stiffness:.2f} ({stiffness_interpret})
        Energy regime: {energy_regime.upper()} - {regime_interpret}
        Critical slowing: {critical_interpret}

      NEURAL ACTIVATIONS (fMRI-like measurements):
        {act_str}

      GRADIENT FLOW (voltage-sensitive dye measurements):
        {grad_str}

      ================================================================================
      REQUIRED ANALYSIS (4-6 technical sentences)
      ================================================================================

      Write a professional analysis following this exact structure:

      1. LEARNING PHASE: State the current phase (early/mid/late/converged) based on loss value and loss trend. Be specific.

      2. REPRESENTATION QUALITY: Evaluate hidden layer health using activation means and dead neuron percentages. State whether representations are diversifying or saturating.

      3. OPTIMIZATION HEALTH: Interpret the stiffness ratio and energy regime. Identify whether training is stable, explosive, or overdamped.

      4. ACTIONABLE RECOMMENDATION: Give ONE specific numerical recommendation (e.g., learning rate change by X%, add gradient clipping, adjust optimizer, etc.).

      RULES:
      - Be quantitative - use the actual numbers provided
      - Do NOT explain what neural networks or XOR are
      - Do NOT use phrases like "based on the data" or "as shown above"
      - Do NOT list options or ask questions
      - Write in complete, declarative sentences

      ANALYSIS:"""
          #print(prompt)
          return prompt

    def _build_prompt_legacy(self, epoch: int, loss: float, activations: Dict, gradients: Dict) -> str:
        """Build structured prompt for LLM - forces detailed response"""
        trend_summary = self.diagnostics.summarize_trends()
        diag = self.diagnostics.get_diagnostics(loss, gradients)

        # Extract activation stats
        act_lines = []
        for name, stats in activations.items():
            act_lines.append(f"  Layer {name}: mean={stats.get('mean', 0):.3f}, dead={stats.get('dead', 0):.2%}")
        act_text = "\n".join(act_lines) if act_lines else "  No activation data"

        # Extract gradient stats
        grad_lines = []
        for name, stats in gradients.items():
            grad_lines.append(f"  Layer {name}: norm={stats.get('norm', 0):.3e}, vanishing={stats.get('vanishing', 0):.2%}")
        grad_text = "\n".join(grad_lines) if grad_lines else "  No gradient data"

        # Detailed prompt that forces longer response
        return f"""You are a computational neuroscientist. Analyze this neural network training snapshot in DETAIL.

DATA:
- Epoch: {epoch}
- Loss: {loss:.6f}
- Loss change: {trend_summary}
- Stiffness: {diag['stiffness']:.2f} (higher = layers learn at different speeds)
- Energy regime: {diag['hamiltonian']['regime']}
- Critical slowing: {diag['critical_slowing']}
- Overall phase: {diag['phase']}

LAYER ACTIVATIONS:
{act_text}

LAYER GRADIENTS:
{grad_text}

Write a DETAILED analysis (8-10 sentences) covering:

1. LEARNING PHASE: Is the network in early, mid, late, or converged phase? Explain why based on loss value.

2. REPRESENTATION QUALITY: Are hidden neurons learning useful features? Discuss activation means and dead neuron percentages.

3. OPTIMIZATION HEALTH: Interpret the stiffness ratio and energy regime. Is training stable, explosive, or overdamped?

4. CRITICAL SLOWING: Is the network stuck or still learning? Explain based on gradient change.

5. ACTIONABLE RECOMMENDATION: What specific hyperparameter change would help? (learning rate, optimizer, architecture)

Be specific, quantitative, and technical. Do NOT explain basic concepts like XOR or neural networks.

ANALYSIS: and give one or two sentences recommendation"""

    def _generate(self, prompt: str, max_tokens: int = 7000) -> str:
        """Generate raw LLM response (no post-processing)"""

        if self.is_seq2seq:
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(self.device)

            outputs = self.model.generate(
              **inputs,
              max_new_tokens=max_tokens,
              do_sample=True,
              temperature=0.9,          # increase exploration
              top_p=0.95,
              num_beams=1,              # REMOVE beam search
              repetition_penalty=1.5,   # reduce from 2.5 (too aggressive)
              no_repeat_ngram_size=2,   # relax constraint
              early_stopping=False,     # IMPORTANT: disable early stop
              min_new_tokens=150        # FORCE longer output
)


            # RAW: no skip_special_tokens, no cleanup
            raw_text = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
            return raw_text

        else:
            messages = [
                {"role": "system", "content": "You are a computational neuroscientist."},
                {"role": "user", "content": prompt}
            ]

            formatted = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = self.tokenizer(
                formatted,
                return_tensors="pt",
                truncation=True,
                max_length=1024
            ).to(self.device)

            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )

            # RAW: includes prompt + generation + special tokens
            raw_text = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
            return raw_text

    def _rule_based_analysis(self, epoch: int, loss: float, activations: Dict, gradients: Dict) -> str:
        """Rule-based fallback analysis"""
        diag = self.diagnostics.get_diagnostics(loss, gradients)

        # Phase determination
        if loss > 0.6:
            phase_text = f"Early learning phase (epoch {epoch}, loss={loss:.4f})"
            recommendation = "Continue training; monitor for rapid loss decrease"
        elif loss > 0.3:
            phase_text = f"Representation formation (epoch {epoch}, loss={loss:.4f})"
            recommendation = f"Monitor convergence; predicted in {diag['t_converge']:.0f} epochs"
        elif loss > 0.05:
            phase_text = f"Converging (epoch {epoch}, loss={loss:.4f})"
            recommendation = "Continue training until loss < 0.05"
        else:
            phase_text = f"Converged (epoch {epoch}, loss={loss:.4f})"
            recommendation = "Training complete - model has learned the task"

        # Energy regime warning
        h = diag['hamiltonian']
        if h['regime'] == "explosive":
            energy_text = f"EXPLOSIVE regime: Energy increasing (ΔH={h['energy_change']:+.4f})"
            recommendation = "REDUCE LEARNING RATE by 50% immediately"
        elif h['regime'] == "overdamped":
            energy_text = f"OVERDAMPED regime: Energy dissipating (K/U={h['kinetic_potential_ratio']:.2f})"
            recommendation = "INCREASE LEARNING RATE or reduce momentum"
        else:
            energy_text = f"CONSERVATIVE regime: Healthy training (ΔH={h['energy_change']:+.4f})"

        # Activation quality
        act_means = [v.get('mean', 0) for v in activations.values() if isinstance(v, dict)]
        if act_means:
            act_summary = f"mean activation {np.mean(act_means):.3f}"
            quality = "good" if np.mean(act_means) > 0.3 else "fair" if np.mean(act_means) > 0.1 else "poor"
        else:
            act_summary = "no activation data"
            quality = "unknown"

        return f"Analysis: {phase_text}. {energy_text}. Hidden layer: {act_summary} ({quality} quality). Recommendation: {recommendation}."


# ============================================================
# CELL 6: Class 4 - NetworkFactory (Neural Network Architectures)
# ============================================================

class NetworkFactory:
    """
    Factory for creating neural network architectures for different tasks.
    Supports XOR, sine wave regression, spirals classification, and parity problems.
    """

    @staticmethod
    def create_xor_network(device: torch.device) -> nn.Module:
        """2-4-1 MLP for XOR problem (baseline)"""
        return nn.Sequential(
            nn.Linear(2, 4),
            nn.Tanh(),
            nn.Linear(4, 1),
            nn.Sigmoid()
        ).to(device)

    @staticmethod
    def create_sine_network(device: torch.device) -> nn.Module:
        """1-64-64-64-1 MLP for sine wave regression (spectral bias testing)"""
        return nn.Sequential(
            nn.Linear(1, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        ).to(device)

    @staticmethod
    def create_spiral_network(device: torch.device) -> nn.Module:
        """2-64-64-1 MLP for two-spirals classification"""
        return nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        ).to(device)

    @staticmethod
    def create_parity_network(n_bits: int = 3, device: torch.device = None) -> nn.Module:
        """n-bit parity network with scaled hidden size"""
        hidden_size = max(8, 2 * n_bits)
        net = nn.Sequential(
            nn.Linear(n_bits, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )
        if device:
            net = net.to(device)
        return net


# ============================================================
# CELL 7: Training Orchestrator (Coordinates Everything)
# ============================================================

class TrainingOrchestrator:
    """
    Orchestrates training, monitoring, and analysis.
    Coordinates the 4 classes: Monitor, Diagnostics, LLM, NetworkFactory.
    """

    def __init__(self, neuroscientist: LLMNeuroscientist):
        self.neuroscientist = neuroscientist
        self.monitor = None

    def train_and_analyze(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        optimizer: torch.optim.Optimizer,
        loss_fn: nn.Module,
        epochs: int = 100,
        log_every: int = 10
    ) -> List[tuple]:
        """
        Train model and analyze at regular intervals.
        Returns list of (epoch, loss, analysis) tuples.
        """
        # Initialize monitor
        self.monitor = NeuralActivityMonitor(model)

        findings = []

        for epoch in range(epochs):
            # Training epoch
            epoch_losses = []
            for x, y in train_loader:
                optimizer.zero_grad()
                out = model(x)
                loss = loss_fn(out, y)
                loss.backward()
                optimizer.step()
                epoch_losses.append(loss.item())

            avg_loss = np.mean(epoch_losses)

            # Analyze at intervals
            if epoch % log_every == 0:
                analysis = self.neuroscientist.analyze(
                    epoch=epoch,
                    loss=avg_loss,
                    activations=self.monitor.activations,
                    gradients=self.monitor.gradients
                )
                findings.append((epoch, avg_loss, analysis))
                print(f"Epoch {epoch:3d} | Loss: {avg_loss:.6f} | Analysis: {analysis[:800]}...")

            # Clear hooks for next epoch
            self.monitor.clear()

        # Clean up
        self.monitor.remove()

        return findings


# ============================================================
# CELL 8: Data Generators (For Different Tasks)
# ============================================================

class DataGenerator:
    """
    Generate datasets for different tasks
    """

    @staticmethod
    def xor_data() -> tuple:
        """XOR dataset (4 samples, 2-bit parity)"""
        X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
        y = torch.tensor([[0.], [1.], [1.], [0.]])
        return X, y

    @staticmethod
    def sine_data(n_samples: int = 1000, noise_std: float = 0.05) -> tuple:
        """Sine wave regression dataset"""
        X = torch.linspace(-3, 3, n_samples).unsqueeze(1)
        y = torch.sin(2 * np.pi * X) + noise_std * torch.randn_like(X)
        return X, y

    @staticmethod
    def spiral_data(n_samples: int = 600, noise: float = 0.12) -> tuple:
        """Two-spirals classification dataset"""
        n = n_samples // 2
        theta = torch.sqrt(torch.rand(n)) * 5 * torch.pi

        # Spiral 1 (class 0)
        r1 = theta
        x1 = r1 * torch.cos(theta) + noise * torch.randn(n)
        y1 = r1 * torch.sin(theta) + noise * torch.randn(n)

        # Spiral 2 (class 1, offset by pi)
        x2 = r1 * torch.cos(theta + torch.pi) + noise * torch.randn(n)
        y2 = r1 * torch.sin(theta + torch.pi) + noise * torch.randn(n)

        X = torch.stack([torch.cat([x1, x2]), torch.cat([y1, y2])]).T
        y = torch.cat([torch.zeros(n), torch.ones(n)]).unsqueeze(1)

        return X, y

    @staticmethod
    def parity_data(n_bits: int = 3) -> tuple:
        """n-bit parity dataset"""
        n_patterns = 2 ** n_bits
        X = torch.zeros(n_patterns, n_bits)
        y = torch.zeros(n_patterns, 1)

        for i in range(n_patterns):
            bits = [(i >> b) & 1 for b in range(n_bits)]
            X[i] = torch.tensor(bits, dtype=torch.float32)
            y[i] = sum(bits) % 2

        return X, y


# ============================================================
# CELL 9: Evaluation Utilities
# ============================================================

def faithfulness_check(text: str, loss: float, gradients: Dict) -> List[str]:
    """Check if LLM analysis contradicts observed data"""
    flags = []

    # Check for false vanishing gradient claims
    if "vanish" in text.lower():
        max_vanishing = max(g.get("vanishing", 0) for g in gradients.values() if isinstance(g, dict))
        if max_vanishing < 0.1:
            flags.append("False vanishing claim")

    # Check for premature convergence claims
    if "conver" in text.lower() and loss > 0.2:
        flags.append("Premature convergence claim")

    # Check for false perfect performance claims
    if "perfect" in text.lower() and loss > 0.05:
        flags.append("False perfect performance claim")

    return flags


def temporal_coherence(text1: str, text2: str) -> float:
    """Compute cosine similarity between sentence embeddings"""
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb1 = model.encode(text1)
    emb2 = model.encode(text2)
    return float(np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2)))


def phase_alignment(text: str, loss: float) -> bool:
    """Check if analysis phase matches loss-based phase"""
    if loss > 0.6 and "early" in text.lower():
        return True
    if 0.3 < loss <= 0.6 and any(word in text.lower() for word in ["mid", "moderate", "representation"]):
        return True
    if loss <= 0.05 and any(word in text.lower() for word in ["converged", "convergence", "final"]):
        return True
    return False


# ============================================================
# CELL 10: Main Execution - Run All Experiments
# ============================================================

def run_experiment(
    name: str,
    data_generator,
    network_creator,
    loss_fn,
    lr: float,
    batch_size: int,
    epochs: int,
    log_every: int,
    llm_model: str = "google/flan-t5-large"
):
    """Run a single experiment with given configuration"""
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {name}")
    print(f"{'='*60}")

    # Generate data
    X, y = data_generator()
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)

    # Initialize components
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    neuroscientist = LLMNeuroscientist(model_name=llm_model)
    orchestrator = TrainingOrchestrator(neuroscientist)

    # Create model
    model = network_creator(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Train and analyze
    results = orchestrator.train_and_analyze(
        model=model,
        train_loader=loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        epochs=epochs,
        log_every=log_every
    )

    return results


# Run all experiments
if __name__ == "__main__":

    # Experiment 1: XOR (Baseline)
    results_xor = run_experiment(
        name="XOR (2-bit parity)",
        data_generator=DataGenerator.xor_data,
        network_creator=NetworkFactory.create_xor_network,
        loss_fn=nn.BCELoss(),
        lr=0.1,
        batch_size=4,
        epochs=100,
        log_every=10,
        llm_model="google/flan-t5-large"
    )

    # Experiment 2: Sine Wave Regression
    results_sine = run_experiment(
        name="Sine Wave Regression",
        data_generator=lambda: DataGenerator.sine_data(n_samples=500),
        network_creator=NetworkFactory.create_sine_network,
        loss_fn=nn.MSELoss(),
        lr=0.001,
        batch_size=32,
        epochs=200,  # Reduced for faster testing
        log_every=20,
        llm_model="google/flan-t5-large"
    )

    # Experiment 3: Two-Spirals Classification
    results_spiral = run_experiment(
        name="Two-Spirals Classification",
        data_generator=DataGenerator.spiral_data,
        network_creator=NetworkFactory.create_spiral_network,
        loss_fn=nn.BCELoss(),
        lr=0.003,
        batch_size=32,
        epochs=200,  # Reduced for faster testing
        log_every=20,
        llm_model="google/flan-t5-large"
    )

    # Print summary
    print("\n" + "="*60)
    print("SUMMARY OF ALL EXPERIMENTS")
    print("="*60)

    all_results = {
        "XOR": results_xor,
        "Sine Wave": results_sine,
        "Two-Spirals": results_spiral
    }

    for name, results in all_results.items():
        if results:
            final_epoch, final_loss, final_analysis = results[-1]
            print(f"\n{name}:")
            print(f"  Final loss: {final_loss:.6f}")
            print(f"  Final analysis: {final_analysis[:150]}...")

    print("\n✅ All experiments completed!")


# ============================================================
# CELL 11: Statistical Validation (Multiple Runs)
# ============================================================

def run_statistical_validation(num_runs: int = 5):
    """Run multiple experiments with different seeds for statistical validation"""

    seeds = [42, 123, 456, 789, 1024]
    all_faithfulness = []
    all_final_losses = []

    for seed in seeds[:num_runs]:
        print(f"\n--- Run with seed {seed} ---")

        # Set seed
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        # Run XOR experiment
        X, y = DataGenerator.xor_data()
        loader = DataLoader(TensorDataset(X, y), batch_size=4, shuffle=True)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        neuroscientist = LLMNeuroscientist(model_name="google/flan-t5-large")
        orchestrator = TrainingOrchestrator(neuroscientist)

        model = NetworkFactory.create_xor_network(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
        loss_fn = nn.BCELoss()

        results = orchestrator.train_and_analyze(
            model=model,
            train_loader=loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            epochs=50,
            log_every=10
        )

        if results:
            final_loss = results[-1][1]
            all_final_losses.append(final_loss)

            # Check faithfulness of last analysis
            flags = faithfulness_check(results[-1][2], final_loss, {})
            all_faithfulness.append(1.0 if len(flags) == 0 else 0.0)

    # Compute statistics
    if all_final_losses:
        print("\n" + "="*60)
        print("STATISTICAL VALIDATION RESULTS")
        print("="*60)
        print(f"Final loss: mean={np.mean(all_final_losses):.6f} \u00b1 {np.std(all_final_losses):.6f}")
        print(f"Faithfulness: mean={np.mean(all_faithfulness):.2f} \u00b1 {np.std(all_faithfulness):.2f}")
        print(f"95% CI loss: {np.mean(all_final_losses) - 1.96*np.std(all_final_losses):.6f} to {np.mean(all_final_losses) + 1.96*np.std(all_final_losses):.6f}")

# Uncomment to run statistical validation
# run_statistical_validation(num_runs=5)


✅ All imports completed

EXPERIMENT: XOR (2-bit parity)


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
/tmp/ipykernel_6069/249934754.py:802: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


✅ Loaded Seq2Seq model: google/flan-t5-large
Epoch   0 | Loss: 0.759778 | Analysis: <pad> Analyzing the performance of a trained CNN with the EPOCH method, we found that training time and energy is influenced by multiple variables, including: Network performance (e.g., L2 averages = 0.8624), current loss (0.827431), convergence prediction (0.499384), energy ratio, neuronal activations, gradient flow, and training conditions (default = Initiating, initializing, Critical slowing, optimization actively exploring landscape). The EPR algorithm has high epoch complexity, which means an efficient and high-quality training set. These findings indicate the success of the training system on multilayer deep learning applications. We also observed good quality of energy regime and the ability to handle critical slowdowns, while minimizing memory transfer for the model layers.</s>...
Epoch  10 | Loss: 0.576231 | Analysis: <pad> The analysis shows that the proposed training architecture is effective

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Loaded Seq2Seq model: google/flan-t5-large
Epoch   0 | Loss: 0.759778 | Analysis: <pad> Analyzing the performance of a trained CNN with the EPOCH method, we found that training time and energy is influenced by multiple variables, including: Network performance (e.g., L2 averages = 0.8624), current loss (0.827431), convergence prediction (0.499384), energy ratio, neuronal activations, gradient flow, and training conditions (default = Initiating, initializing, Critical slowing, optimization actively exploring landscape). The EPR algorithm has high epoch complexity, which means an efficient and high-quality training set. These findings indicate the success of the training system on multilayer deep learning applications. We also observed good quality of energy regime and the ability to handle critical slowdowns, while minimizing memory transfer for the model layers.</s>...
Epoch  10 | Loss: 0.576231 | Analysis: <pad> The analysis shows that the proposed training architecture is effective

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Loaded Seq2Seq model: google/flan-t5-large
Epoch   0 | Loss: 0.759778 | Analysis: <pad> Analyzing the performance of a trained CNN with the EPOCH method, we found that training time and energy is influenced by multiple variables, including: Network performance (e.g., L2 averages = 0.8624), current loss (0.827431), convergence prediction (0.499384), energy ratio, neuronal activations, gradient flow, and training conditions (default = Initiating, initializing, Critical slowing, optimization actively exploring landscape). The EPR algorithm has high epoch complexity, which means an efficient and high-quality training set. These findings indicate the success of the training system on multilayer deep learning applications. We also observed good quality of energy regime and the ability to handle critical slowdowns, while minimizing memory transfer for the model layers.</s>...
Epoch  10 | Loss: 0.576231 | Analysis: <pad> The analysis shows that the proposed training architecture is effective